# Phase 4b — Agent with Spatial Refinement

Same as the previous agent notebook, but candidate routes are now differentiated using **two blended signals**:

1. **Regional LSTM forecast** — captures the temporal trend (is this region's turbulence rising or falling over the last few hours)
2. **Local spatial risk** — distance-weighted average of actual nearby PIREPs (within ~150nm of the exact waypoint), which varies continuously with location instead of jumping only at region boundaries

Why this instead of just using more/smaller regions: subdividing the grid further would starve each cell of data (we already have limited PIREPs). Blending in spatial interpolation gives fine-grained differentiation without needing more training data or retraining the LSTM.

## Step 1 — Setup (same as before)

In [ ]:
import requests
import pandas as pd
import numpy as np
import pickle
import math
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/flight_corridor_project'

import tensorflow as tf
model = tf.keras.models.load_model(f'{SAVE_DIR}/turbulence_lstm.keras')

with open(f'{SAVE_DIR}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open(f'{SAVE_DIR}/region_columns.pkl', 'rb') as f:
    region_columns = pickle.load(f)

SEQ_LEN = 6

SUPABASE_URL = "........"   # <-- same as before
SUPABASE_KEY = "..........."                    # <-- same as before

headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
}

print("Model, scaler, and region encoding loaded.")

Mounted at /content/drive
Model, scaler, and region encoding loaded.


## Step 2 — Region geometry + distance math (same as before)

In [ ]:
region_boxes = {
    "NE":  (35, -90, 50, -65),
    "SE":  (24, -90, 35, -65),
    "MW":  (35, -105, 50, -90),
    "SW":  (24, -105, 35, -90),
    "NW":  (35, -125, 50, -105),
    "SWC": (24, -125, 35, -105),
}

def get_region(lat, lon):
    for name, (minLat, minLon, maxLat, maxLon) in region_boxes.items():
        if minLat <= lat <= maxLat and minLon <= lon <= maxLon:
            return name
    return None

def haversine_nm(lat1, lon1, lat2, lon2):
    R_nm = 3440.065
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlambda/2)**2
    return 2 * R_nm * math.asin(math.sqrt(a))

## Step 3 — Pull recent data (raw + hourly aggregate)

We now keep **both**: the hourly-aggregated version (for the LSTM, same as before) and the **raw point-level** version (for spatial interpolation, new).

In [ ]:
def fetch_recent_pireps(hours_back=24, page_size=1000):
    cutoff_ts = int((pd.Timestamp.utcnow() - pd.Timedelta(hours=hours_back)).timestamp())
    all_rows = []
    offset = 0
    while True:
        params = {
            "select": "*", "order": "obs_time.asc",
            "limit": page_size, "offset": offset,
            "obs_time": f"gte.{cutoff_ts}"
        }
        resp = requests.get(f"{SUPABASE_URL}/rest/v1/pireps", headers=headers, params=params)
        if resp.status_code != 200:
            print("Fetch failed:", resp.status_code, resp.text[:300])
            break
        batch = resp.json()
        if not batch:
            break
        all_rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return pd.DataFrame(all_rows)

recent_df = fetch_recent_pireps(hours_back=24)
print("Recent raw rows pulled:", len(recent_df))

if len(recent_df) > 0:
    recent_df["obs_datetime"] = pd.to_datetime(recent_df["obs_time"], unit="s")
    recent_df["region"] = recent_df.apply(lambda r: get_region(r["lat"], r["lon"]), axis=1)
    recent_df = recent_df.dropna(subset=["region", "lat", "lon", "turb_score"])
    recent_df["hour"] = recent_df["obs_datetime"].dt.floor("h")
    recent_hourly = recent_df.groupby(["region", "hour"])["turb_score"].max().reset_index()
else:
    recent_hourly = pd.DataFrame(columns=["region", "hour", "turb_score"])
    print("No recent data — falling back to zero-risk sequences.")

Recent raw rows pulled: 1891


## Step 4 — The two risk signals

**Signal A — LSTM regional forecast** (unchanged from before)

**Signal B — local spatial risk** (new): looks at actual nearby PIREPs, weighting closer reports more heavily using a Gaussian-style decay. A report right at the waypoint counts fully; one 150nm away barely counts; beyond ~300nm it's effectively ignored.

In [ ]:
def get_recent_sequence(region, seq_len=SEQ_LEN):
    sub = recent_hourly[recent_hourly["region"] == region].sort_values("hour")
    scores = sub["turb_score"].values[-seq_len:] if len(sub) > 0 else np.array([])
    if len(scores) < seq_len:
        pad = np.zeros(seq_len - len(scores))
        scores = np.concatenate([pad, scores])
    scaled = scaler.transform(pd.DataFrame(scores, columns=["turb_score"])).flatten()
    return scaled

def predict_region_risk(region):
    seq = get_recent_sequence(region)
    region_onehot = [1.0 if col == f"region_{region}" else 0.0 for col in region_columns]
    seq_features = np.array([[s] + region_onehot for s in seq]).reshape(1, SEQ_LEN, -1)
    pred_scaled = model.predict(seq_features, verbose=0)[0][0]
    pred_scaled = np.clip(pred_scaled, 0, 1)
    pred_score = scaler.inverse_transform(pd.DataFrame([[pred_scaled]], columns=["turb_score"]))[0][0]
    return max(0.0, pred_score)

def local_spatial_risk(lat, lon, sigma_nm=75, max_radius_nm=300):
    """Distance-weighted average turbulence score from nearby recent PIREPs."""
    if len(recent_df) == 0:
        return 0.0

    distances = recent_df.apply(lambda r: haversine_nm(lat, lon, r["lat"], r["lon"]), axis=1)
    nearby = recent_df[distances <= max_radius_nm]
    nearby_dist = distances[distances <= max_radius_nm]

    if len(nearby) == 0:
        return 0.0

    weights = np.exp(-(nearby_dist.values ** 2) / (2 * sigma_nm ** 2))
    if weights.sum() < 1e-6:
        return 0.0

    return float(np.average(nearby["turb_score"].values, weights=weights))

def waypoint_risk(lat, lon, region_weight=0.5, spatial_weight=0.5):
    """Blend the regional LSTM forecast with local spatial interpolation."""
    region = get_region(lat, lon)
    lstm_risk = predict_region_risk(region) if region else 0.0
    spatial_risk = local_spatial_risk(lat, lon)
    return region_weight * lstm_risk + spatial_weight * spatial_risk

# Quick test: risk at a few specific points, not just region averages
test_points = [(40.6, -73.8), (38.5, -90.0), (34.0, -100.0)]
for lat, lon in test_points:
    r = waypoint_risk(lat, lon)
    print(f"({lat}, {lon}) -> blended risk: {r:.2f} / 8  [region: {get_region(lat, lon)}]")

(40.6, -73.8) -> blended risk: 1.84 / 8  [region: NE]
(38.5, -90.0) -> blended risk: 1.83 / 8  [region: NE]
(34.0, -100.0) -> blended risk: 1.20 / 8  [region: SW]


## Step 5 — Route generation (same as before)

In [ ]:
def generate_waypoints(origin, destination, n_points=5):
    lat1, lon1 = origin
    lat2, lon2 = destination
    return [
        (lat1 + (lat2 - lat1) * t, lon1 + (lon2 - lon1) * t)
        for t in np.linspace(0, 1, n_points)
    ]

def shift_route(waypoints, lat_offset):
    shifted = [waypoints[0]]
    for lat, lon in waypoints[1:-1]:
        shifted.append((lat + lat_offset, lon))
    shifted.append(waypoints[-1])
    return shifted

def route_extra_distance_nm(original, shifted):
    def path_length(wpts):
        return sum(haversine_nm(wpts[i][0], wpts[i][1], wpts[i+1][0], wpts[i+1][1]) for i in range(len(wpts)-1))
    return path_length(shifted) - path_length(original)

## Step 6 — Evaluate routes using the blended per-waypoint risk

The key change from before: instead of `max(risk per region hit)`, we now compute `max(risk per individual waypoint)` using the blended signal — so two routes in the same region but at different exact locations can now score differently.

In [ ]:
FUEL_KG_PER_NM = 6.5
SPEED_KTS = 470

def evaluate_route(label, waypoints, original_waypoints):
    waypoint_risks = [waypoint_risk(lat, lon) for lat, lon in waypoints]
    risk = max(waypoint_risks)

    extra_nm = route_extra_distance_nm(original_waypoints, waypoints)
    extra_fuel_kg = max(0, extra_nm) * FUEL_KG_PER_NM
    extra_time_min = max(0, extra_nm) / SPEED_KTS * 60

    return {
        "label": label,
        "predicted_risk": round(risk, 2),
        "waypoint_risks": [round(r, 2) for r in waypoint_risks],
        "extra_distance_nm": round(extra_nm, 1),
        "extra_fuel_kg": round(extra_fuel_kg, 1),
        "extra_time_min": round(extra_time_min, 1),
    }

def rank_candidates(origin, destination, deviations_deg=(0, 1, -1, 2, -2, 3, -3, 4, -4)):
    original_wpts = generate_waypoints(origin, destination)
    results = []
    for dev in deviations_deg:
        label = "Direct route" if dev == 0 else f"{'North' if dev > 0 else 'South'} deviation ({abs(dev)}° lat)"
        candidate_wpts = shift_route(original_wpts, dev) if dev != 0 else original_wpts
        results.append(evaluate_route(label, candidate_wpts, original_wpts))

    RISK_WEIGHT = 10
    COST_WEIGHT = 0.01
    for r in results:
        r["composite_score"] = RISK_WEIGHT * r["predicted_risk"] + COST_WEIGHT * r["extra_fuel_kg"]

    results.sort(key=lambda r: r["composite_score"])
    return results

## Step 7 — Run it

In [ ]:
JFK = (40.6413, -73.7781)
LAX = (33.9416, -118.4085)

candidates = rank_candidates(JFK, LAX)

for c in candidates:
    print(f"{c['label']:30s} | risk: {c['predicted_risk']:.2f}/8 | "
          f"+{c['extra_distance_nm']:.0f} nm | +{c['extra_fuel_kg']:.0f} kg | "
          f"+{c['extra_time_min']:.1f} min | waypoint risks: {c['waypoint_risks']}")

best = candidates[0]
print(f"\n Recommended: {best['label']}")

North deviation (2° lat)       | risk: 1.87/8 | +-18 nm | +0 kg | +0.0 min | waypoint risks: [np.float32(1.84), np.float32(1.76), np.float32(1.87), np.float32(1.83), np.float32(0.87)]
North deviation (3° lat)       | risk: 1.95/8 | +-9 nm | +0 kg | +0.0 min | waypoint risks: [np.float32(1.84), np.float32(1.73), np.float32(1.74), np.float32(1.95), np.float32(0.87)]
North deviation (1° lat)       | risk: 1.98/8 | +-15 nm | +0 kg | +0.0 min | waypoint risks: [np.float32(1.84), np.float32(1.76), np.float32(1.98), np.float32(1.81), np.float32(0.87)]
Direct route                   | risk: 2.04/8 | +0 nm | +0 kg | +0.0 min | waypoint risks: [np.float32(1.84), np.float32(1.81), np.float32(1.9), np.float32(2.04), np.float32(0.87)]
North deviation (4° lat)       | risk: 1.98/8 | +12 nm | +76 kg | +1.5 min | waypoint risks: [np.float32(1.84), np.float32(1.67), np.float32(1.63), np.float32(1.98), np.float32(0.87)]
South deviation (1° lat)       | risk: 2.22/8 | +28 nm | +180 kg | +3.5 min | waypoi

## Step 8 — Explanation (same template as before)

In [ ]:
def explain_recommendation(candidates):
    best = candidates[0]
    direct = next((c for c in candidates if c["label"] == "Direct route"), None)

    lines = []
    lines.append(f"RECOMMENDATION: {best['label']}")
    lines.append(f"Predicted turbulence risk: {best['predicted_risk']:.1f} / 8")

    if direct and best["label"] != "Direct route":
        risk_reduction = direct["predicted_risk"] - best["predicted_risk"]
        if risk_reduction > 0:
            lines.append(
                f"This reduces predicted turbulence risk by {risk_reduction:.1f} points vs. the direct route, "
                f"at a cost of {best['extra_distance_nm']:.0f} nm (+{best['extra_fuel_kg']:.0f} kg fuel, "
                f"+{best['extra_time_min']:.1f} min)."
            )
    elif best["label"] == "Direct route":
        lines.append("The direct route has the lowest predicted risk — no deviation recommended.")

    lines.append("\nAll evaluated options:")
    for c in candidates:
        lines.append(f"  • {c['label']}: risk {c['predicted_risk']:.1f}/8, +{c['extra_fuel_kg']:.0f} kg, +{c['extra_time_min']:.1f} min")

    lines.append("\nNote: this is a decision-support recommendation only. Any route change must be requested from and approved by ATC.")
    return "\n".join(lines)

print(explain_recommendation(candidates))

RECOMMENDATION: North deviation (2° lat)
Predicted turbulence risk: 1.9 / 8
This reduces predicted turbulence risk by 0.2 points vs. the direct route, at a cost of -18 nm (+0 kg fuel, +0.0 min).

All evaluated options:
  • North deviation (2° lat): risk 1.9/8, +0 kg, +0.0 min
  • North deviation (3° lat): risk 2.0/8, +0 kg, +0.0 min
  • North deviation (1° lat): risk 2.0/8, +0 kg, +0.0 min
  • Direct route: risk 2.0/8, +0 kg, +0.0 min
  • North deviation (4° lat): risk 2.0/8, +76 kg, +1.5 min
  • South deviation (1° lat): risk 2.2/8, +180 kg, +3.5 min
  • South deviation (2° lat): risk 2.2/8, +438 kg, +8.6 min
  • South deviation (3° lat): risk 2.0/8, +767 kg, +15.1 min
  • South deviation (4° lat): risk 1.9/8, +1162 kg, +22.8 min

Note: this is a decision-support recommendation only. Any route change must be requested from and approved by ATC.


## Summary

Candidates should now show **different** risk scores even within the same broad region, since each waypoint's risk blends the regional LSTM trend with actual nearby report locations.

Tunable knobs if you want to experiment:
- `region_weight` / `spatial_weight` in `waypoint_risk()` — how much to trust the temporal model vs. the spatial snapshot
- `sigma_nm` in `local_spatial_risk()` — how quickly influence fades with distance
- `RISK_WEIGHT` / `COST_WEIGHT` in `rank_candidates()` — how safety-conscious vs. fuel-conscious the agent is

